# Exploratory Data Analysis — NASA CMAPSS FD001

This notebook walks through the turbofan engine degradation dataset to build
intuition about sensor behavior, failure patterns, and the characteristics
that a predictive model will need to capture.

**Dataset:** C-MAPSS (Commercial Modular Aero-Propulsion System Simulation) — FD001 subset.  
**Conditions:** Single operating condition, single fault mode (HPC degradation).  
**Source:** NASA Prognostics Center / Kaggle.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATASET_DIR, ALL_COLS, INDEX_COLS, SETTING_COLS, SENSOR_COLS, RUL_CAP

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')

## 1. Loading the raw data

The text files are whitespace-separated with no header row. Each row represents
one operating cycle of one engine.

In [ ]:
train = pd.read_csv(DATASET_DIR / 'train_FD001.txt', sep=r'\s+', header=None)
train.columns = ALL_COLS

test = pd.read_csv(DATASET_DIR / 'test_FD001.txt', sep=r'\s+', header=None)
test.columns = ALL_COLS

rul = pd.read_csv(DATASET_DIR / 'RUL_FD001.txt', sep=r'\s+', header=None, names=['rul'])

print(f'Training set: {train.shape[0]:,} rows, {train["engine_id"].nunique()} engines')
print(f'Test set:     {test.shape[0]:,} rows, {test["engine_id"].nunique()} engines')
print(f'RUL labels:   {rul.shape[0]} entries')

In [ ]:
train.head()

## 2. Engine lifespan distribution

Since the training engines are run-to-failure, the maximum cycle number for each
engine tells us its total lifespan. Let's see how that varies.

In [ ]:
lifespans = train.groupby('engine_id')['cycle'].max()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lifespans, bins=30, edgecolor='white', color='#4c72b0')
ax.axvline(lifespans.median(), color='red', linestyle='--', label=f'Median = {lifespans.median():.0f}')
ax.set_xlabel('Total Cycles (Lifespan)')
ax.set_ylabel('Number of Engines')
ax.set_title('Engine Lifespan Distribution — FD001 Training Set')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Lifespan range: {lifespans.min()} – {lifespans.max()} cycles')
print(f'Mean: {lifespans.mean():.1f}, Median: {lifespans.median():.1f}, Std: {lifespans.std():.1f}')

**Observation:** Lifespans range from roughly 130 to 360 cycles. There's a healthy spread,
which means the model genuinely needs to learn degradation patterns rather than memorizing
a fixed failure point. The median sits around 200 cycles — that gives us a reasonable baseline
to beat.

## 3. RUL target distribution

We clip RUL at 125 because predicting exact remaining life when an engine is essentially
brand new isn't particularly useful — and it makes the loss landscape smoother.

In [ ]:
max_cycles = train.groupby('engine_id')['cycle'].max().reset_index()
max_cycles.columns = ['engine_id', 'max_cycle']
train_rul = train.merge(max_cycles, on='engine_id')
train_rul['rul'] = (train_rul['max_cycle'] - train_rul['cycle']).clip(upper=RUL_CAP)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_rul['rul'], bins=50, edgecolor='white', color='#55a868')
ax.set_xlabel('RUL (cycles)')
ax.set_ylabel('Count')
ax.set_title('RUL Distribution (capped at 125)')
plt.tight_layout()
plt.show()

**Observation:** With the cap, there's a large cluster at RUL = 125 (healthy engines) and a
roughly uniform spread below that. The cap prevents the model from fitting a long tail of
high-RUL values that carry little practical meaning.

## 4. Sensor behavior over time

Let's look at all 21 sensors for a single engine to see which ones show clear
degradation trends and which ones are essentially constant.

In [ ]:
engine_1 = train[train['engine_id'] == 1]

fig, axes = plt.subplots(7, 3, figsize=(16, 20), sharex=True)
axes = axes.ravel()

for i, sensor in enumerate(SENSOR_COLS):
    axes[i].plot(engine_1['cycle'], engine_1[sensor], linewidth=0.7, color='#2c7fb8')
    axes[i].set_title(sensor, fontsize=10)
    axes[i].grid(True, alpha=0.3)

fig.suptitle('All 21 Sensors — Engine #1', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

**Observation:** Sensors 1, 5, 6, 10, 16, 18, and 19 are basically flat lines — they
contain no useful information about degradation. Sensors like 2, 3, 4, 7, 8, 11, 12, 13,
15, and 21 show clear upward or downward trends as the engine approaches failure.
These are the ones our model should pay attention to.

## 5. Identifying constant sensors

A more systematic way to confirm: compute the coefficient of variation for each sensor.

In [ ]:
sensor_stats = train[SENSOR_COLS].describe().T[['mean', 'std']]
sensor_stats['cv'] = sensor_stats['std'] / sensor_stats['mean'].abs().clip(lower=1e-8)
sensor_stats = sensor_stats.sort_values('cv')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#c0392b' if cv < 0.01 else '#27ae60' for cv in sensor_stats['cv']]
ax.barh(sensor_stats.index, sensor_stats['cv'], color=colors, edgecolor='white')
ax.set_xlabel('Coefficient of Variation')
ax.set_title('Sensor Variability — Red = nearly constant (to be dropped)')
ax.axvline(0.01, color='black', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

low_var = sensor_stats[sensor_stats['cv'] < 0.01].index.tolist()
print(f'Sensors to drop (CV < 0.01): {low_var}')

## 6. Correlation heatmap

Let's see how the sensors relate to each other and to RUL.

In [ ]:
corr_data = train_rul[SENSOR_COLS + ['rul']].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(corr_data, mask=mask, annot=False, cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Sensor Correlation Heatmap (including RUL)', fontsize=14)
plt.tight_layout()
plt.show()

**Observation:** There are strong correlation clusters — sensors 2, 3, 4, 11, and 17
are highly correlated with each other. Sensor 7, 8, 12, 13, 15, and 21 form another
cluster. This multicollinearity isn't a problem for neural networks (they handle it
naturally), but it explains why we can drop some sensors without losing much.

## 7. Correlation with RUL

Which sensors are most predictive of remaining useful life?

In [ ]:
rul_corr = train_rul[SENSOR_COLS + ['rul']].corr()['rul'].drop('rul').sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in rul_corr]
rul_corr.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Pearson Correlation with RUL')
ax.set_title('Sensor Correlation with Remaining Useful Life')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** Sensors 2, 3, 4, 8, 11, 13, and 17 have the strongest linear
relationship with RUL. The flat sensors (1, 5, 6, etc.) show near-zero correlation,
confirming that dropping them won't hurt model performance.

## 8. Multi-engine overlay

Let's overlay sensor degradation for multiple engines to see how consistent
the failure pattern is.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
key_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12']

for ax, sensor in zip(axes.ravel(), key_sensors):
    for eid in range(1, 11):  # first 10 engines
        eng = train[train['engine_id'] == eid]
        ax.plot(eng['cycle'], eng[sensor], alpha=0.5, linewidth=0.7)
    ax.set_title(sensor)
    ax.set_xlabel('Cycle')
    ax.grid(True, alpha=0.3)

fig.suptitle('Key Sensor Trends — Engines 1–10', fontsize=14)
fig.tight_layout()
plt.show()

**Observation:** The degradation trajectories are consistent in direction across engines,
but the *rate* and *starting point* differ. This is exactly why a sequence model (LSTM)
is well-suited — it can track the gradient of change rather than relying on absolute values.

## 9. Settings analysis

In [ ]:
print('Unique values per setting column:')
for col in SETTING_COLS:
    print(f'  {col}: {train[col].nunique()} unique values, range [{train[col].min():.4f}, {train[col].max():.4f}]')

print(f'\nFD001 is a single operating condition dataset, so settings are nearly constant.')
print('setting_3 has zero variance and can be dropped.')

## 10. Summary

| Aspect | Finding |
|---|---|
| **Engines** | 100 (train), 100 (test) |
| **Lifespan** | 128–362 cycles, median ~200 |
| **Constant sensors** | 1, 5, 6, 10, 16, 18, 19 — dropped |
| **Most predictive** | 2, 3, 4, 7, 8, 11, 12, 13, 15, 17, 21 |
| **RUL cap** | 125 (piecewise linear) |
| **Operating condition** | Single (FD001) |

The data is clean, well-structured, and shows clear degradation signals. The main
modeling challenge is capturing the *temporal dynamics* of degradation — not the
presence/absence of signal — which justifies the CNN-LSTM-Attention architecture.